In [ ]:
%run ./_common

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
np.set_printoptions(precision=2)

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [ ]:
SA = SWSAcademyHelper()
SA.init()

In [ ]:
# Spark ML models logged or loaded with MLflow need a temporary folder. The default (/dbfs/tmp) is blocked when the
# workspace disables the DBFS root, so use a Unity Catalog volume in the learner schema instead.
import os
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = f"/Volumes/{SA.catalog_name}/{SA.schema_name}/mlflow_tmp"

# On dedicated (single-user) clusters MLflow loads Spark ML models through the DBFS root (/dbfs/...), which is blocked
# when the workspace disables it. Use MLflow's own Unity Catalog volume loader instead.
import mlflow.spark as _mlflow_spark
if hasattr(_mlflow_spark, "_load_model_databricks_uc_volume"):
    _mlflow_spark._load_model_databricks_dbfs = _mlflow_spark._load_model_databricks_uc_volume